# Nested final-set decision-policy screen

This notebook keeps the pool-50/checkpoint-1208/`enhanced_v2` ranking frozen. It reads saved development OOF rankings and compares leakage-aware final-set policies only. It does not rerun retrieval/reranking and does not read public or private labels.

In [ ]:
from pathlib import Path
import shutil, subprocess, tarfile

REPO = Path('/kaggle/working/Drill-reliable-retrieval')
WORK = Path('/kaggle/working/final_decision_policy_screen')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/minhnhtl/Drill-reliable-retrieval.git', str(REPO)], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', f'{REPO}[ltr]'], check=True)
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(parents=True)
print({'repo': str(REPO), 'work': str(WORK)})

In [ ]:
def unique(paths, label):
    paths = sorted(set(paths))
    if len(paths) != 1: raise RuntimeError(f'Expected exactly one {label}; found {paths}')
    return paths[0]

archive = unique(Path('/kaggle/input').rglob('pool50_epoch1_lightgbm_feature_screen.tar.gz'), 'feature-screen archive')
source = WORK / 'feature_screen'
source.mkdir()
with tarfile.open(archive, 'r:gz') as bundle: bundle.extractall(source, filter='data')
oof = source / 'lightgbm_enhanced_v2' / 'oof_predictions.jsonl'
dev_qids = source / 'selected_reranker_dev_run' / 'split_dev_qids.json'
existing_metrics = source / 'lightgbm_enhanced_v2' / 'metrics.json'
preferred_queries = Path('/kaggle/input/datasets/lmnhnht125/wedev-vlsp/train.parquet')
queries = preferred_queries if preferred_queries.is_file() else unique([p for p in Path('/kaggle/input').rglob('train.parquet') if p.parent.name.lower() == 'wedev-vlsp'], 'WeDev-VLSP training query parquet')
for path in (oof, dev_qids, existing_metrics, queries):
    if not path.is_file(): raise FileNotFoundError(path)
print({'archive': str(archive), 'oof': str(oof), 'dev_qids': str(dev_qids), 'queries': str(queries)})

In [ ]:
OUTPUT = WORK / 'results'
command = [
    'python', 'scripts/evaluate_decision_policies.py',
    '--oof-predictions', str(oof), '--queries', str(queries),
    '--dev-qids', str(dev_qids), '--existing-lightgbm-metrics', str(existing_metrics),
    '--output-dir', str(OUTPUT),
    '--folds', '5', '--seed', '42', '--fallback-k', '2', '--maximum-k', '5',
]
print('RUN:', ' '.join(command))
subprocess.run(command, cwd=REPO, check=True)

In [ ]:
import json, pandas as pd

scoreboard = pd.read_csv(OUTPUT / 'decision_policy_scoreboard.csv').sort_values('f2', ascending=False)
display(scoreboard)
metrics = json.loads((OUTPUT / 'decision_policy_metrics.json').read_text())
display(pd.DataFrame([{'policy': name, **{k: row[k] for k in ('precision','recall','f2')}} for name, row in metrics['policies'].items()]).sort_values('f2', ascending=False))
bundle = Path('/kaggle/working/final_decision_policy_screen.tar.gz')
with tarfile.open(bundle, 'w:gz') as archive_out:
    archive_out.add(OUTPUT, arcname='final_decision_policy_screen')
print({'artifact': str(bundle), 'bytes': bundle.stat().st_size})